In [1]:
%load_ext autoreload
%autoreload 2
%load_ext rpy2.ipython

In [2]:
import ibis
import pandas as pd

import src
from src.load import DataLoader

ibis.options.interactive = True
r_colormap_channel = src.r_colormap_name
r_colormap_party = src.r_colormap_party
r_out = str(src.OUT)
pd.options.display.float_format = "{:.1f}".format

In [4]:
%%R -i r_colormap_channel -i r_colormap_party -i r_out

suppressMessages(library(tidyverse))
library(ggplot2)
library(ggeffects)
library(here)
library(ggpubr)

options(scipen = 999)

cmap <- setNames(r_colormap_channel$color, r_colormap_channel$channel)
pmap <- setNames(r_colormap_party$color, r_colormap_party$party)

# Load Data

In [ ]:
dl = DataLoader()

videos = dl.channels().join(dl.videos(filtered=True), "channel_id").to_pandas()

sentences = dl.sentences(filtered=True).join(dl.popbert(filtered=False), "sentence_id").to_pandas()

sents = sentences.groupby("video_id", observed=True).agg(
    n_sents=("video_id", "size"),
    n_elite=("elite", "sum"),
    n_pplcentr=("pplcentr", "sum"),
    avg_elite=("elite", "mean"),
    avg_pplcentr=("pplcentr", "mean"),
)

# Dataset Summary Table

In [ ]:
channel_overview = (
    videos.loc[videos.video_was_live == False]
    .merge(sents, on="video_id", how="left")
    .groupby("channel", observed=True)
    .agg(
        chFollowers=("channel_follower_count", "first"),
        nShortVids=("video_is_short", lambda x: (x == 1).sum()),
        nLongVids=("video_is_short", lambda x: (x == 0).sum()),
        nVids=("video_was_live", lambda x: (x == 0).sum()),
        nLikesNA=("video_likes", lambda x: x.isna().sum()),
        meanViews=("video_views", "mean"),
        medianViews=("video_views", "median"),
        meanLikes=("video_likes", "mean"),
        medianLikes=("video_likes", "median"),
        meanVidLen=("video_duration", "mean"),
        nSentences=("n_sents", "sum"),
        first_video=("video_datetime_upload", "min"),
        latest_video=("video_datetime_upload", "max"),
    )
)

In [ ]:
inlines = src.OUT / "manuscript/inlines"
inlines.mkdir(exist_ok=True)

In [ ]:
# number of broken transcripts

broken_transcripts = dl.broken_transcripts(filtered=True).to_pandas()
n_broken = len(broken_transcripts)

n_broken = f"{n_broken:,.0f}"
p = inlines / "n_broken.txt"
p.unlink(missing_ok=True)
p.write_text(n_broken)
print(f"faulty transcripts: {n_broken}")

In [ ]:
# sum of durations

sum_of_seconds = videos.video_duration.sum()
hour_duration = f"{round(sum_of_seconds / 60 / 60, 1):,.1f}"
p = inlines / "sum_duration.txt"
p.unlink(missing_ok=True)
p.write_text(hour_duration)
print(f"Total sum of video durations: {hour_duration} hours")

In [ ]:
# number of videos

count_videos = len(videos)
count_videos = f"{count_videos:,.0f}"
p = inlines / "n_videos.txt"
p.unlink(missing_ok=True)
p.write_text(count_videos)
print(f"Total number of valid videos: {count_videos}")

In [ ]:
# number of sentencs

count_sents = len(sentences)
count_sents = f"{count_sents:,.0f}"
p = inlines / "n_sents.txt"
p.unlink(missing_ok=True)
p.write_text(count_sents)
print(f"Total number of valid sentences: {count_sents}")

In [ ]:
summary_table = channel_overview.drop(["first_video", "latest_video"], axis=1)

summary_table

In [ ]:
path = src.OUT / "tables/videos_per_channel.csv"
path.unlink(missing_ok=True)
summary_table[
    [
        "chFollowers",
        "nShortVids",
        "nLongVids",
        "nVids",
        "meanVidLen",
        "nSentences",
    ]
].to_csv(path)

In [ ]:
path = src.OUT / "tables/channel_descriptives.csv"
path.unlink(missing_ok=True)
summary_table[["meanViews", "medianViews", "meanLikes", "medianLikes"]].to_csv(path)

# View Count Violin Plot

In [ ]:
df = videos.merge(sents, on="video_id")

In [ ]:
%%R -i df -w 1000 -h 800

df_plot <- df |>
   mutate(
      likes = video_likes + 1,
      views = video_views + 1,
) |>
pivot_longer(
   c(likes, views),
   names_to = "var",
   values_to = "val"
)

plot <- ggplot(df_plot, aes(x=channel, y=val, fill=channel)) +
   geom_boxplot(alpha=0.6) +
   geom_violin(alpha=0.3, trim=T, scale="width") +
   scale_y_continuous(trans="log10", breaks=scales::breaks_log(n=8)) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 21
   ) +
   theme(
      axis.text.x=element_text(angle=90, hjust=1),
      axis.ticks.x=element_blank(),
      axis.title.x=element_blank(),
      legend.position = "none"
   ) +
   facet_wrap(~var, ncol=1, scales="free_y") +
   ylab("log(10)")

plot

p <- here(r_out, "/figures/view_count.svg")
if (file.exists(p)) file.remove(p)
ggsave(p, width=12.4, height=12.4)
plot

# Populism Amount Plot

In [ ]:
%%R -i df -w 1000 -h 800

df_plot <- df |>
   mutate(
      elite = (n_elite / n_sents * 100) + 1,
      pplcentr = (n_pplcentr / n_sents * 100) + 1,
) |>
pivot_longer(
   c(elite, pplcentr),
   names_to = "var",
   values_to = "val"
)


plot <- ggplot(df_plot, aes(x=channel, y=val, fill=channel)) +
   geom_boxplot(alpha=0.6, outliers=F, coef=0.5) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=90, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   facet_wrap(~var, ncol=1, scales="free_y")

plot

p <- here(r_out, "/figures/populism_per_party.svg")
if (file.exists(p)) file.remove(p)
ggsave(p, width=12.4, height=12.4)
plot